In [ ]:
import numpy as np
np.random.default_rng(2159)

PYTHONHASHSEED=0

In [ ]:
%matplotlib inline

#load packages
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import os

sc.settings.verbosity = 3   
sc.set_figure_params(scanpy=True, fontsize=12, dpi_save=180, format='svg', transparent=True)
sc.settings.set_figure_params(dpi=80)

In [ ]:
my_c = ['#fe864a', '#c9d179', '#95d0fc', '#fbdd7e', '#388004', '#c760ff', '#76cd26', '#ffbacd', '#d6b4fc',
        '#cfaf7b','#8b88f8', '#acc2d9']

# Load raw counts

In [ ]:
adata_filt = sc.read_h5ad('Postnatal_noDBL_filtered.h5ad')

In [ ]:
adata_raw = sc.read_h5ad('Postnatal_Postnatal_unfiltered.h5ad')

In [ ]:
adata_annotated = sc.read_h5ad('Postnatal_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
adata_raw.var_names_make_unique()

adata_annotated.var_names_make_unique()

In [ ]:
adata_all = adata_raw[adata_filt.obs.index]

In [ ]:
adata_all.obs = adata_filt.obs

In [ ]:
adata_all

# Add metadata

In [ ]:
adata_all.obs['Cell_type'] = adata_annotated.obs['Cell_type']

# Subset data only for Epithelial cells

In [ ]:
adata_all

In [ ]:
adata = adata_all[adata_all.obs['Cell_type'].isin(['Epithelial cells'])]

In [ ]:
adata

In [ ]:
adata.write_h5ad('Epithelial_Postnatal_unfiltered.h5ad')

In [ ]:
adata = sc.read_h5ad('Epithelial_Postnatal_unfiltered.h5ad')

In [ ]:
adata

## Ploting data before QC

In [ ]:
sc.pl.violin(adata, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb'],
             ncols=2, rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_epithelial_rawdata.svg')

In [ ]:
sc.pl.violin(adata, ["Xist-counts", "percent_chrY"], jitter=0, groupby = 'batch', rotation= 90, palette=my_c)

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)

In [ ]:
%matplotlib inline


sc.pl.umap(adata, color=['Gender', 'Xist-counts'], vmax=1, palette=my_c)

In [ ]:
%matplotlib inline


sc.pl.umap(adata, color=['Gender', 'percent_chrY'], vmax=0.1)

In [ ]:
fig, (ax1,ax2) = plt.subplots(nrows=2,ncols=1,constrained_layout=True,figsize=(13,12))
sns.countplot(y = 'batch',data = adata.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)
    ax1.set_ylabel('Cells in each sample')

sns.countplot(y = 'Age',data = adata.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)
    ax2.set_ylabel('Cells in each time point')

plt.show()
    
fig.savefig('epithelial_barplots_rawdata.svg', format='svg', dpi=180)

In [ ]:
sc.pl.highest_expr_genes(adata,n_top=20)

In [ ]:
sc.pl.violin(adata, ['S_score', 'G2M_score'], jitter=0, groupby = 'batch', rotation=90, palette=my_c)

In [ ]:
adata

# QC

## Data normalization and logaritmization

In [ ]:
## normalize to depth 10 000
sc.pp.normalize_per_cell(adata,counts_per_cell_after=1e4)

## logaritmize
sc.pp.log1p(adata)

**Creating raw data object**

In [ ]:
adata.raw = adata.copy()

## Searching for doublets

In [ ]:
##  Predict doublets
import scrublet as scr

scrub = scr.Scrublet(adata.raw.X)
doublet_score, predicted_doublets = scrub.scrub_doublets()

In [ ]:
%matplotlib inline

In [ ]:
scrub.plot_histogram()

In [ ]:
adata.obs['predicted_doublets'] = predicted_doublets

In [ ]:
sum(adata.obs['predicted_doublets'])

In [ ]:
## add in column with singlet/doublet instead of True/False
adata.obs['doublet_info'] = adata.obs["predicted_doublets"].astype(str)
sc.pl.violin(adata, 'n_genes_by_counts', jitter=0.4, groupby = 'doublet_info', rotation=45, palette=my_c, 
             save='_epithelial_doublets.svg')

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)

In [ ]:
%matplotlib inline


sc.pl.umap(adata, color=['doublet_info'], palette=my_c, save='_epithelial_doublets.svg')

In [ ]:
adata = adata[adata.obs['doublet_info'] == 'False',:]
print(adata.shape)

In [ ]:
adata

## Saving data without doublets

In [ ]:
adata.write_h5ad('Epithelial_Postnatal_noDBL.h5ad')

In [ ]:
adata = sc.read_h5ad('Epithelial_Postnatal_noDBL.h5ad')

In [ ]:
adata

## Filtering cells and genes

In [ ]:
sc.pp.filter_cells(adata,min_genes = 250)
sc.pp.filter_genes(adata,min_cells = 3)

In [ ]:
%matplotlib inline

sc.pl.violin(adata, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_epithelial_prefilter.svg')

In [ ]:
adata = adata[adata.obs['total_counts'] < 80000,:] 

In [ ]:
adata = adata[adata.obs['n_genes_by_counts'] < 8100, :]

In [ ]:
adata = adata[adata.obs['pct_counts_Mito'] < 20,:]

In [ ]:
adata = adata[adata.obs['pct_counts_Hb'] < 0.03,:]

In [ ]:
adata = adata[adata.obs['pct_counts_Ribo'] < 20,:]

In [ ]:
adata = adata[adata.obs['pct_counts_Ribo'] > 2.5,:]

In [ ]:
%matplotlib inline

sc.pl.violin(adata, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_epithelial_afterfilter.svg')

In [ ]:
print("Remaining cells %d" % adata.n_obs)

In [ ]:
adata

In [ ]:

fig, (ax1,ax2) = plt.subplots(nrows=2,ncols=1,constrained_layout=True,figsize=(13,12))
sns.countplot(y = 'batch',data = adata.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)
    ax1.set_ylabel('Cells in each sample')

sns.countplot(y = 'Age',data = adata.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)
    ax2.set_ylabel('Cells in each time point')
    
    
fig.savefig('epithelial_barplots_afterfilter.svg', format='svg')

## Saving filtered data

In [ ]:
adata.write_h5ad('Epithelial_Postnatal_noDBL_filtered.h5ad')

## Loading filtered data

In [ ]:
adata = sc.read_h5ad('Epithelial_Postnatal_noDBL_filtered.h5ad')

In [ ]:
adata

In [ ]:
adata

# Dimensionality redutcion

## HVG

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

In [ ]:
sc.pl.highly_variable_genes(adata, save='_epithelial_HVG.svg')

In [ ]:
var_genes_all = adata.var.highly_variable

In [ ]:
adata = adata[:, var_genes_all]

In [ ]:
adata

## PCA

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca(adata, color='batch', components = ['1,2','3,4','5,6','7,8'], ncols=2, wspace=0.8, palette=my_c, 
          save='_epithelial_pca.svg')

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50, save='_epithelial_n_pcs.svg')

# Clustering

In [ ]:
sc.pp.neighbors(adata, n_pcs=26, n_neighbors=15)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=["batch", "Age"], size=1, wspace=0.8, ncols=2, palette=my_c, 
           save='_epithelial_precorrection.svg')

## Batch correction

### Harmony

In [ ]:
from harmony import harmonize

In [ ]:
Z = harmonize(adata.obsm['X_pca'], adata.obs, batch_key = 'batch', max_iter_harmony=20)
adata.obsm['X_harmony'] = Z

In [ ]:
print("Calculating UMAP")
sc.pp.neighbors(adata, n_pcs=26, n_neighbors=15, use_rep="X_harmony")
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color=["batch", "Age"], size=1, wspace=0.8, ncols=2, palette=my_c, 
           save='_epithelial_aftercorrection.svg')

## Clustering

In [ ]:
# clustering
for res in [3.0]:
    sc.tl.leiden(adata, resolution=res,key_added = 'leiden_%s'%res)

In [ ]:
#plot
sc.pl.umap(adata,color=['leiden_0.8', 'leiden_1.0', 'leiden_1.2', 'leiden_1.8', 'leiden_2.0', 'leiden_2.8', 
                        'leiden_3.0'], ncols=4, wspace=1, cmap='my_c')

## Saving clsutered data

In [ ]:
adata.write_h5ad('Epithelial_Postnatal_noDBL_filtered_clustered.h5ad')

## Loading filtered data

In [ ]:
adata = sc.read_h5ad('Epithelial_Postnatal_noDBL_filtered_clustered.h5ad')

In [ ]:
adata

### Leiden_0.1

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.1', method='wilcoxon', key_added = "wilcoxon_01", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_01" , groupby="leiden_0.1", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.2

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.2', method='wilcoxon', key_added = "wilcoxon_02", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_02" , groupby="leiden_0.2", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.3

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.3', method='wilcoxon', key_added = "wilcoxon_03", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_03" , groupby="leiden_0.3", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.4

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.4', method='wilcoxon', key_added = "wilcoxon_04", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_04" , groupby="leiden_0.4", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.5

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.5', method='wilcoxon', key_added = "wilcoxon_05", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_05" , groupby="leiden_0.5", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.6

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.6', method='wilcoxon', key_added = "wilcoxon_06", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_06" , groupby="leiden_0.6", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.8

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_0.8', method='wilcoxon', key_added = "wilcoxon_08", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_08" , groupby="leiden_0.8", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.0

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_1.0', method='wilcoxon', key_added = "wilcoxon_1", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_1" , groupby="leiden_1.0", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.2

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_1.2', method='wilcoxon', key_added = "wilcoxon_12", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_12" , groupby="leiden_1.2", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.4

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_1.4', method='wilcoxon', key_added = "wilcoxon_14", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_14" , groupby="leiden_1.4", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.6

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_1.6', method='wilcoxon', key_added = "wilcoxon_16", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_16" , groupby="leiden_1.6", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.8

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_1.8', method='wilcoxon', key_added = "wilcoxon_18", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_18" , groupby="leiden_1.8", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_2.0

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden_2.0', method='wilcoxon', key_added = "wilcoxon_2", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_2" , groupby="leiden_2.0", cmap="YlGnBu", vmin=0, vmax=10)

## Saving clsutered data

In [ ]:
adata.write_h5ad('Epithelial_Postnatal_noDBL_filtered_clustered.h5ad')

## Loading clustered data

In [ ]:
adata = sc.read_h5ad('Epithelial_Postnatal_noDBL_filtered_clustered.h5ad')

In [ ]:
adata

In [ ]:
adata

In [ ]:
adata_all = sc.read_h5ad('Epithelial_Postnatal_unfiltered.h5ad')

In [ ]:
markers = {"Epithelial cells": ['Pon1', 'Gata3','Atp10b','Cidea', "Krt18","Htr2c","Car12","Clic6", "Folr1","Igfbp2",'Otx2','Chchd10','Clu','Calml4'],
           "Mesenchymal cells": ['Pdgfrb','Mgp','Cfh','Dcn','Tgfbi','Moxd1','Cp','Bgn','Cxcl12','Col3a1', 'Flt1'],
           "Endothelial cells": ["Egfl7","Kdr","Pecam1", "Cd34", "Plvap",'Cdh5','Flt1','Igfbp3','Ly6a','Ly6c1'],
           "Immune cells": ["C1qa",'Tyrobp','Fcer1g','Ctss','C1qb','Ptprc','C1qc','Cd83','Lyz2','Laptm5'],
           "NeuroGlia": ['Tubb2b','Gpm6b','Rtn1','Olig1','Fabp7','Bcan','Cnp','Scrg1','Opcml','Pcsk1n'], 
           "NIPC": ["Cdca7", "2810417H13Rik", "Rrm2", "Ube2c", "Top2a", "Birc5", "P2rx3"]}

In [ ]:
for k, v in markers.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_all.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=5)

In [ ]:
markers1 = {'Ciliogenesis': ['Shisa8', 'Ccdc67', 'Ccno', 'Mcidas'], 
                'Mesenchymal cells': ['Dcn', 'Adam12', 'Cxcl12', 'Lum'], 
                'Epithelial cells': ['Htr2c', 'Enpp2', 'Kcnj13', 'Aqp1', 'Gata3'], 
                'Endothelial cells': ['Plvap', 'Flt1', 'Pecam1'],
                'Neuro / Glial': ['Slc1a3', 'Olig1', 'Olig2', 'Dlx1', 'Dlx2', 'Eomes', 'Tbr2'],
                'Macrophages': ['Cx3cr1', 'Mrc1', 'C1qc', 'Cfsr1'], 
                'Lymphocytes': ['Cd3g', 'Cxcr6', 'Txk', 'Tcf7', 'Tnfrsf25'], 
                'Monocytes': ['Cd14', 'Cd44', 'Ccr2', 'Wfdc17', 'Ms4a4c', 'Sirpb1c'], 
                'Mast cells': ['Kit', 'Smpx', 'Clec12b'], 
                'B cells': ['Cd79a', 'Cd19', 'Vpreb3', 'Il7r'], 
                'Basophils': ['Gata2', 'Fcer1a2', 'Cd200r3', 'Csf1'],
                'Neutrophils': ['S110a9', 'Ly6g', 'Ly6ca'], 
                'DC': ['Cd74', 'H2.Ab1', 'H2.Aa'], 
                'Undefined': ['Gadd45b', 'Ddit3', 'Htra2', 'Nr1d1'], 
                'EMT': ['Cdh2', 'Cdh11', 'Itga5', 'Itgb6', 'Sdc1', 'Fsp1', 'Vim', 'Ctnnb1', 'Fn1', 'Lama5', 'Col1a1', 'Snail1', 'Zeb1', 'Foxc2', 'Gsc']} 

In [ ]:
for k, v in markers1.items():
    genes_in_data = []
    for gene in v:
        if gene in adata.raw.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=5)

In [ ]:
markers2 = {'Neuroblasts': ['Tubb3', 'Dcx', 'Islr2'], 
            'TAPs': ['Mki67', 'Ube2c', 'Top2a'], 
            'NSCs': ['Helt', 'Cdk6', 'Mcm2'],
            'OPCs': ['Pdgfra', 'Cdo1', 'Lhfpl3', 'Snap25'], 
            'Neurons': ['Pcp4', 'Meg3', 'Nrgn', 'Snap25'],
            'Ependymal cells': ['Enkur', 'Foxj1', 'Ccdc153'],
            'Astrocytes': ['Slc1a3', 'S1pr1', 'Cldn10'], 
            'Microglia, activated': ['Ccl4', 'Egr1', 'Fos', 'Fosb'],
            'Endothelial cells': ['Flt1', 'Egfl7', 'Cldn5'],
            'Oligodendrocytes': ['Enpp2', 'Mog', 'Opalin', 'Sox10', 'Plp1', 'Mog', 'Mag', 'Car2'],
            'transition cells': ['Ascl1', 'Egfr', 'Notum', 'S100a6'],
            'COP': ['Gpr17', 'Tnr', 'Fyn']}

In [ ]:
for k, v in markers2.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_all.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=8)

In [ ]:
adata

# Removing doublets

In [ ]:
adata_edit = adata[(adata.obs['leiden_2.8']!='20')]

adata_noDBL = adata_all[adata_edit.obs.index]

In [ ]:
adata_noDBL.write_h5ad('noDBL_Epithelial_Postnatal_unfiltered.h5ad')

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Epithelial_Postnatal_unfiltered.h5ad')

In [ ]:
adata_noDBL

## Ploting data before QC

In [ ]:
sc.pl.violin(adata_noDBL, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb'],
             ncols=2, rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_epithelial_noDBL_rawdata.svg')

In [ ]:
sc.pl.violin(adata_noDBL, ["Xist-counts", "percent_chrY"], jitter=0, groupby = 'batch', rotation= 90, palette=my_c)

In [ ]:
sc.tl.pca(adata_noDBL, svd_solver='arpack')
sc.pp.neighbors(adata_noDBL, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_noDBL)

In [ ]:
%matplotlib inline


sc.pl.umap(adata_noDBL, color=['Gender', 'Xist-counts'], vmax=1, palette=my_c)

In [ ]:
%matplotlib inline


sc.pl.umap(adata_noDBL, color=['Gender', 'percent_chrY'], vmax=0.1)

In [ ]:
fig, (ax1,ax2) = plt.subplots(nrows=2,ncols=1,constrained_layout=True,figsize=(18,12))
sns.countplot(y = 'batch',data = adata_noDBL.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)
    ax1.set_ylabel('Cells in each sample')

sns.countplot(y = 'Age',data = adata_noDBL.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)
    ax2.set_ylabel('Cells in each time point')
    
    
fig.savefig('barplots_epithelial_noDBL_rawdata.svg')

In [ ]:
sc.pl.highest_expr_genes(adata_noDBL,n_top=20)

In [ ]:
sc.pl.violin(adata_noDBL, ['S_score', 'G2M_score'], jitter=0, groupby = 'batch', rotation=90, palette=my_c)

In [ ]:
adata_noDBL

# QC

## Data normalization and logaritmization

In [ ]:
## normalize to depth 10 000
sc.pp.normalize_per_cell(adata_noDBL,counts_per_cell_after=1e4)

## logaritmize
sc.pp.log1p(adata_noDBL)

**Creating raw data object**

In [ ]:
adata_noDBL.raw = adata_noDBL.copy()

## Searching for doublets

In [ ]:
##  Predict doublets
import scrublet as scr

scrub = scr.Scrublet(adata_noDBL.raw.X)
doublet_score, predicted_doublets = scrub.scrub_doublets()

In [ ]:
%matplotlib inline

In [ ]:
scrub.plot_histogram()

In [ ]:
adata_noDBL.obs['predicted_doublets'] = predicted_doublets

In [ ]:
sum(adata_noDBL.obs['predicted_doublets'])

In [ ]:
## add in column with singlet/doublet instead of True/False
adata_noDBL.obs['doublet_info'] = adata_noDBL.obs["predicted_doublets"].astype(str)
sc.pl.violin(adata_noDBL, 'n_genes_by_counts', jitter=0.4, groupby = 'doublet_info', rotation=45, palette=my_c, 
             save='_epithelial_noDBL_doublets.svg')

In [ ]:
sc.tl.pca(adata_noDBL, svd_solver='arpack')
sc.pp.neighbors(adata_noDBL, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_noDBL)

In [ ]:
%matplotlib inline


sc.pl.umap(adata_noDBL, color=['doublet_info'], palette=my_c, save='_epithelial_noDBL_doublets.svg')

In [ ]:
adata_noDBL = adata_noDBL[adata_noDBL.obs['doublet_info'] == 'False',:]
print(adata_noDBL.shape)

In [ ]:
adata_noDBL

## Saving data without doublets

In [ ]:
adata_noDBL.write_h5ad('noDBL_Epithelial_Postnatal_noDBL.h5ad')

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Epithelial_Postnatal_noDBL.h5ad')

In [ ]:
adata_noDBL

## Filtering cells and genes

In [ ]:
sc.pp.filter_cells(adata_noDBL,min_genes = 250)
sc.pp.filter_genes(adata_noDBL,min_cells = 3)

In [ ]:
%matplotlib inline

sc.pl.violin(adata_noDBL, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_epithelial_noDBL_prefilter.svg')

In [ ]:
adata_noDBL = adata_noDBL[adata_noDBL.obs['total_counts'] < 80000,:] 

In [ ]:
adata_noDBL = adata_noDBL[adata_noDBL.obs['n_genes_by_counts'] < 8000, :]

In [ ]:
adata_noDBL = adata_noDBL[adata_noDBL.obs['pct_counts_Mito'] < 20,:]

In [ ]:
adata_noDBL = adata_noDBL[adata_noDBL.obs['pct_counts_Hb'] < 0.025,:]

In [ ]:
adata_noDBL = adata_noDBL[adata_noDBL.obs['pct_counts_Ribo'] < 18,:]

In [ ]:
adata_noDBL = adata_noDBL[adata_noDBL.obs['pct_counts_Ribo'] > 2.5,:]

In [ ]:
%matplotlib inline

sc.pl.violin(adata_noDBL, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_epithelial_noDBL_afterfilter.svg')

In [ ]:
print("Remaining cells %d" % adata_noDBL.n_obs)

In [ ]:
adata_noDBL

In [ ]:
fig, (ax1,ax2) = plt.subplots(nrows=2,ncols=1,constrained_layout=True,figsize=(13,12))
sns.countplot(y = 'batch',data = adata_noDBL.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)
    ax1.set_ylabel('Cells in each sample')

sns.countplot(y = 'Age',data = adata_noDBL.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)
    ax2.set_ylabel('Cells in each time point')
    
fig.savefig('barplots_epithelial_noDBL_afterfilter.svg', format='svg')

## Saving filtered data

In [ ]:
adata_noDBL.write_h5ad('noDBL_Epithelial_Postnatal_noDBL_filtered.h5ad')

## Loading filtered data

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Epithelial_Postnatal_noDBL_filtered.h5ad')

In [ ]:
adata_noDBL

In [ ]:
adata_noDBL

# Dimensionality redutcion

## HVG

In [ ]:
#HVGs
sc.pp.highly_variable_genes(adata_noDBL, min_mean=0.0125, max_mean=3, min_disp=0.5, flavor='seurat_v3', n_top_genes=4000)

In [ ]:
adata_noDBL

In [ ]:
sc.pl.highly_variable_genes(adata_noDBL, save='_epithelial_noDBL_HVG.svg')

In [ ]:
var_genes_all = adata_noDBL.var.highly_variable

In [ ]:
adata_noDBL = adata_noDBL[:, var_genes_all]

In [ ]:
adata_noDBL

In [ ]:
sc.pp.scale(adata_noDBL,max_value=10)

## PCA

In [ ]:
sc.tl.pca(adata_noDBL, svd_solver='arpack')

In [ ]:
sc.pl.pca(adata_noDBL, color='batch', components = ['1,2','3,4','5,6','7,8'], ncols=2, wspace=0.8, palette=my_c, 
          save='_epithelial_noDBL_pca.svg')

In [ ]:
sc.pl.pca_variance_ratio(adata_noDBL, log=True, n_pcs=50, save='_epithelial_noDBL_n_pcs.svg')

# Clustering

In [ ]:
sc.pp.neighbors(adata_noDBL, n_pcs=32, n_neighbors=15)
sc.tl.umap(adata_noDBL)

In [ ]:
sc.pl.umap(adata_noDBL, color=["batch", "Age"], size=1, wspace=0.8, ncols=2, palette=my_c, 
           save='_epithelial_noDBL_precorrection.svg')

## Batch correction

### Harmony

In [ ]:
from harmony import harmonize

In [ ]:
Z = harmonize(adata_noDBL.obsm['X_pca'], adata_noDBL.obs, batch_key = 'batch', max_iter_harmony=20)
adata_noDBL.obsm['X_harmony'] = Z

In [ ]:
print("Calculating UMAP")
sc.pp.neighbors(adata_noDBL, n_pcs=32, n_neighbors=15, use_rep="X_harmony")
sc.tl.umap(adata_noDBL)

In [ ]:
print("Ploting UMAP")
sc.pl.umap(adata_noDBL, color=["batch", 'Age'], size=1, wspace=0.8, ncols=2, palette=my_c, 
           save='_epithelial_noDBL_aftercorrection.svg')

## Clustering

In [ ]:
# clustering
for res in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]:
    sc.tl.leiden(adata_noDBL, resolution=res,key_added = 'leiden_%s'%res)

In [ ]:
#plot
sc.pl.umap(adata_noDBL,color=['leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'leiden_0.6', 
                        'leiden_0.8', 'leiden_1.0', 'leiden_1.2', 'leiden_1.4', 'leiden_1.6', 'leiden_1.8', 
                        'leiden_2.0'], ncols=4, wspace=1, cmap='my_c')

## Saving clsutered data

In [ ]:
adata_noDBL.write_h5ad('noDBL_Epithelial_Postnatal_noDBL_filtered_clustered.h5ad')

## Loading filtered data

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Epithelial_Postnatal_noDBL_filtered_clustered.h5ad')

In [ ]:
adata_noDBL

In [ ]:
adata_noDBL


### Leiden_0.1

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.1', method='wilcoxon', key_added = "wilcoxon_01", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_01" , groupby="leiden_0.1", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.2

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.2', method='wilcoxon', key_added = "wilcoxon_02", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_02" , groupby="leiden_0.2", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.3

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.3', method='wilcoxon', key_added = "wilcoxon_03", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_03" , groupby="leiden_0.3", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.4

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.4', method='wilcoxon', key_added = "wilcoxon_04", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_04" , groupby="leiden_0.4", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.5

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.5', method='wilcoxon', key_added = "wilcoxon_05", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_05" , groupby="leiden_0.5", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.6

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.6', method='wilcoxon', key_added = "wilcoxon_06", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_06" , groupby="leiden_0.6", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.8

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_0.8', method='wilcoxon', key_added = "wilcoxon_08", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_08" , groupby="leiden_0.8", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.0

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_1.0', method='wilcoxon', key_added = "wilcoxon_1", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_1" , groupby="leiden_1.0", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.2

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_1.2', method='wilcoxon', key_added = "wilcoxon_12", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_12" , groupby="leiden_1.2", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.4

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_1.4', method='wilcoxon', key_added = "wilcoxon_14", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_14" , groupby="leiden_1.4", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.6

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_1.6', method='wilcoxon', key_added = "wilcoxon_16", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_16" , groupby="leiden_1.6", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.8

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_1.8', method='wilcoxon', key_added = "wilcoxon_18", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_18" , groupby="leiden_1.8", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_2.0

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'leiden_2.0', method='wilcoxon', key_added = "wilcoxon_2", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_2" , groupby="leiden_2.0", cmap="YlGnBu", vmin=0, vmax=10)

## Saving clsutered data

In [ ]:
adata_noDBL.write_h5ad('noDBL_Epithelial_Postnatal_noDBL_filtered_clustered.h5ad')

## Loading clustered data

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Epithelial_Postnatal_noDBL_filtered_clustered.h5ad')

In [ ]:
adata_noDBL

In [ ]:
adata_noDBL

In [ ]:
adata_noDBL_all = sc.read_h5ad('noDBL_Epithelial_Postnatal_unfiltered.h5ad')

In [ ]:
markers = {"Epithelial cells": ['Gata3','Atp10b','Cidea', "Krt18","Htr2c","Car12","Clic6", "Folr1","Igfbp2",'Otx2','Chchd10','Clu','Calml4'],
           "Mesenchymal cells": ['Pdgfrb','Mgp','Cfh','Dcn','Tgfbi','Moxd1','Cp','Bgn','Cxcl12','Col3a1'],
           "Endothelial cells": ["Egfl7","Kdr","Pecam1", "Cd34", "Plvap",'Cdh5','Flt1','Igfbp3','Ly6a','Ly6c1'],
           "Immune cells": ["C1qa",'Tyrobp','Fcer1g','Ctss','C1qb','Ptprc','C1qc','Cd83','Lyz2','Laptm5'],
           "NeuroGlia": ['Tubb2b','Gpm6b','Rtn1','Olig1','Fabp7','Bcan','Cnp','Scrg1','Opcml','Pcsk1n'], 
           "NIPC": ["Cdca7", "2810417H13Rik", "Rrm2", "Ube2c", "Top2a", "Birc5", "P2rx3"]}

In [ ]:
for k, v in markers.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_noDBL_all.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata_noDBL, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=6)

In [ ]:
markers1 = {'Ciliogenesis': ['Shisa8', 'Ccdc67', 'Ccno', 'Mcidas'], 
                'Mesenchymal cells': ['Dcn', 'Adam12', 'Cxcl12', 'Lum'], 
                'Epithelial cells': ['Htr2c', 'Enpp2', 'Kcnj13', 'Aqp1', 'Gata3'], 
                'Endothelial cells': ['Plvap', 'Flt1', 'Pecam1'],
                'Neuro / Glial': ['Slc1a3', 'Olig1', 'Olig2', 'Dlx1', 'Dlx2', 'Eomes', 'Tbr2'],
                'Macrophages': ['Cx3cr1', 'Mrc1', 'C1qc', 'Cfsr1'], 
                'Lymphocytes': ['Cd3g', 'Cxcr6', 'Txk', 'Tcf7', 'Tnfrsf25'], 
                'Monocytes': ['Cd14', 'Cd44', 'Ccr2', 'Wfdc17', 'Ms4a4c', 'Sirpb1c'], 
                'Mast cells': ['Kit', 'Smpx', 'Clec12b'], 
                'B cells': ['Cd79a', 'Cd19', 'Vpreb3', 'Il7r'], 
                'Basophils': ['Gata2', 'Fcer1a2', 'Cd200r3', 'Csf1'],
                'Neutrophils': ['S110a9', 'Ly6g', 'Ly6ca'], 
                'DC': ['Cd74', 'H2.Ab1', 'H2.Aa'], 
                'Undefined': ['Gadd45b', 'Ddit3', 'Htra2', 'Nr1d1'], 
                'EMT': ['Cdh2', 'Cdh11', 'Itga5', 'Itgb6', 'Sdc1', 'Fsp1', 'Vim', 'Ctnnb1', 'Fn1', 'Lama5', 'Col1a1', 'Snail1', 'Zeb1', 'Foxc2', 'Gsc']} 

In [ ]:
for k, v in markers1.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_noDBL.raw.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata_noDBL, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=5)

In [ ]:
markers2 = {'Neuroblasts': ['Tubb3', 'Dcx', 'Islr2'], 
            'TAPs': ['Mki67', 'Ube2c', 'Top2a'], 
            'NSCs': ['Helt', 'Cdk6', 'Mcm2'],
            'OPCs': ['Pdgfra', 'Cdo1', 'Lhfpl3', 'Snap25'], 
            'Neurons': ['Pcp4', 'Meg3', 'Nrgn', 'Snap25'],
            'Ependymal cells': ['Enkur', 'Foxj1', 'Ccdc153'],
            'Astrocytes': ['Slc1a3', 'S1pr1', 'Cldn10'], 
            'Microglia, activated': ['Ccl4', 'Egr1', 'Fos', 'Fosb'],
            'Endothelial cells': ['Flt1', 'Egfl7', 'Cldn5'],
            'Oligodendrocytes': ['Enpp2', 'Mog', 'Opalin', 'Sox10', 'Plp1', 'Mog', 'Mag', 'Car2'],
            'transition cells': ['Ascl1', 'Egfr', 'Notum', 'S100a6'],
            'COP': ['Gpr17', 'Tnr', 'Fyn']}

In [ ]:
for k, v in markers2.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_noDBL_all.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata_noDBL, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=8)

In [ ]:
adata_noDBL

In [ ]:
adata_noDBL

# Cluster annotation

In [ ]:
adata_noDBL.obs["Cell_type"] = adata_noDBL.obs["leiden_1.8"].replace({"0": "Epithelial cells",
                                                                      "14": "Epithelial cells", 
                                                                      "16": "Epithelial cells Gata3+",
                                                                      "1": "Epithelial cells",
                                                                      "9": "Epithelial cells",
                                                                      "6": "Epithelial cells",
                                                                      "4": "Epithelial cells", 
                                                                      "7": "Epithelial cells",
                                                                      "13": "Epithelial cells",
                                                                      "19": "Epithelial cells",
                                                                      "20": "Epithelial cells",
                                                                      "21": "Ciliogenesis",
                                                                      "25": "Epithelial cells proliferating",
                                                                      "12": "Epithelial cells",
                                                                      "15": "Epithelial cells", 
                                                                      "17": "Epithelial cells", 
                                                                      "10": "Epithelial cells",
                                                                      "27": "Epithelial cells",
                                                                      "23": "Epithelial cells", 
                                                                      "22": "Epithelial cells Gpx3+",
                                                                      "24": "Epithelial cells",
                                                                      "18": "Epithelial cells",
                                                                      "28": "Epithelial cells",
                                                                      "3": "Epithelial cells", 
                                                                      "29": "Epithelial cells", 
                                                                      "5": "Epithelial cells", 
                                                                      "8": "Epithelial cells", 
                                                                      "11": "Epithelial cells", 
                                                                      "2": "Epithelial cells", 
                                                                      "26": "Epithelial cells"}) 


In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, 'Cell_type', method='wilcoxon', key_added = "wilcoxon_Cell_type", use_raw=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_Cell_type" , groupby="Cell_type", 
                                cmap="YlGnBu", vmin=0, vmax=10, save='_epithelial_noDBL_15_nofiltered.svg')

In [ ]:
sc.tl.filter_rank_genes_groups(adata_noDBL, groupby='Cell_type', key = "wilcoxon_Cell_type", 
                               key_added='wilcoxon_filtered_Cell_type',
                               min_in_group_fraction=0.1,max_out_group_fraction=0.5,min_fold_change=0.25)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=15, key="wilcoxon_filtered_Cell_type" , groupby="Cell_type", 
                                cmap="YlGnBu", vmin=0, vmax=5, save='_epithelial_filtered_15.svg')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_noDBL, n_genes=10, key="wilcoxon_filtered_Cell_type" , groupby="Cell_type", 
                                cmap="YlGnBu", vmin=0, vmax=5, save='_epithelial_filtered_10.svg')

In [ ]:
sc.pl.umap(adata_noDBL, color=['Cell_type', 'Age', 'batch'], ncols=4, cmap='YlGnBu',wspace=0.7, palette=my_c,
           save='_epithelial_celltypes_age_batch.svg')

In [ ]:
adata_noDBL.obs['P2/P3'] = False
adata_noDBL.obs['P11'] = False
adata_noDBL.obs['P60'] = False

In [ ]:
ages = ['P2/P3', 'P11', 'P60']

In [ ]:
for p in ages:
    for cell in adata_noDBL.obs_names:
        if adata_noDBL.obs['Age'].loc[cell] == p:
            adata_noDBL.obs[p].loc[cell] = p
        else:
            adata_noDBL.obs[p].loc[cell] = np.nan

In [ ]:
sc.pl.umap(adata_noDBL, color=ages, size=1, wspace=0.5, cmap='YlGnBu_r', na_color="#FF000000", 
           save='_epithelial_ages.svg')

In [ ]:
sc.tl.rank_genes_groups(adata_noDBL, groupby="Cell_type", method='wilcoxon', n_genes=50)

clust_df = pd.DataFrame(adata_noDBL.uns['rank_genes_groups']['names'])

In [ ]:
clust_df.to_csv("Epithelial_Top_50_Postnatal_mouse_marker_genes_CP.csv")

In [ ]:
markers = ['Zfp949', 'Gpx3', 'Fam227b', 'Cp',
           'Deup1', 'Cdc20b', 'Ccno', 'Shisa8',
           'Birc5', 'Lockd', 'Mki67', 'Nusap1',
           'Ttr', 'Folr1', 'Clic6', 'Otx2',
           'Pon1', 'Sgcd', 'Gata3', 'Atp10b']

In [ ]:
sns.set_style({'axes.facecolor': 'white',
 'axes.edgecolor': '.8',
 'axes.grid': False,
 'axes.axisbelow': 'line',
 'axes.labelcolor': 'black',
 'figure.facecolor': 'white',
 'grid.color': '.8',
 'grid.linestyle': '-',
 'text.color': 'black',
 'xtick.color': 'black',
 'ytick.color': 'black',
 'xtick.direction': 'out',
 'ytick.direction': 'out',
 'patch.edgecolor': 'black',
 'patch.force_edgecolor': False,
 'image.cmap': 'viridis',
 'font.family': ['sans-serif'],
 'font.sans-serif': ['DejaVu Sans',
  'Bitstream Vera Sans',
  'Computer Modern Sans Serif',
  'Lucida Grande',
  'Verdana',
  'Geneva',
  'Lucid',
  'Arial',
  'Helvetica',
  'Avant Garde',
  'sans-serif'],
 'xtick.bottom': True,
 'xtick.top': False,
 'ytick.left': True,
 'ytick.right': False,
 'axes.spines.left': True,
 'axes.spines.bottom': True,
 'axes.spines.right': True,
 'axes.spines.top': True}
)

vp = sc.pl.rank_genes_groups_stacked_violin(adata_noDBL, var_names=markers, groupby='Cell_type', key='wilcoxon_Cell_type', categories_order=cells,
                                            row_palette=color_order, dendrogram=False,  yticklabels = True, return_fig=True, figsize=(13,5))

vp.style(ylim=(0,10)).show()

vp.savefig('stacked_violin_NEW_noDBL_celltype_markers_epithelial_cells.svg', format='svg', dpi=180)
axes_dict = vp.get_axes()
print(axes_dict)

In [ ]:
markers = {"Epithelial cells Gpx3+": ['Zfp949', 'Gpx3', 'Fam227b', 'Cp'],
           "Ciliogenesis": ['Deup1', 'Cdc20b', 'Ccno', 'Shisa8'],
           "Epithelial cells proliferating": ['Birc5', 'Lockd', 'Mki67', 'Nusap1'],
           "Epithelial cells": ['Ttr', 'Folr1', 'Clic6', 'Otx2'],
           "Epithelial cells Gata3+": ['Pon1', 'Sgcd', 'Gata3', 'Atp10b']}

In [ ]:
for k, v in markers.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_noDBL_all.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata_noDBL, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=4, 
               save=f"_epithelial_{k}_marker_genes_all.svg")

In [ ]:
sc.pl.violin(adata_noDBL, ['total_counts','n_genes_by_counts','pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb'], 
             groupby="Cell_type", rotation=90, stripplot=False, palette=my_c, save='_epithelial_celltypes_metrics.svg')

In [ ]:
sc.pl.violin(adata_noDBL, ['total_counts','n_genes_log','pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb'], 
             groupby="Cell_type", rotation=90, stripplot=False, palette=my_c, 
             save='_epithelial_celltypes_metrics_genes_log.svg')

In [ ]:
fig, (ax1,ax2,ax3) = plt.subplots(nrows=3,ncols=1,constrained_layout=True,figsize=(16,12))
sns.countplot(y = 'batch',data = adata_noDBL.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)

sns.countplot(y = 'Cell_type',data = adata_noDBL.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)

sns.countplot(y = 'Age',data = adata_noDBL.obs,ax=ax3, palette=my_c)
for container in ax3.containers:
    ax3.bar_label(container)

fig.savefig('barplots_epithelial_celltypes.svg', format='svg')

In [ ]:
def batch_pct_cellType(adata_noDBL_file,groupby_key,legend_key,pos):
    tmp = pd.crosstab(adata_noDBL_file.obs[groupby_key],adata_noDBL_file.obs[legend_key], normalize='index')   
    tmp.plot.bar(stacked=True,rot= 90, grid=False, color=my_c,
                 xlabel='',ax=pos).legend(bbox_to_anchor=[1.15,1])

In [ ]:
fig, (ax1,ax2) = plt.subplots(2, 1, constrained_layout=True,figsize=(8,10))
batch_pct_cellType(adata_noDBL,'Cell_type','Age',ax1)
batch_pct_cellType(adata_noDBL,'Cell_type','batch',ax2)

fig.savefig('barplots_epithelial_celltypes_age.svg', format='svg')

### Cell proportions

In [ ]:
ct = pd.crosstab(adata_noDBL.obs['batch'],adata_noDBL.obs['Cell_type'], normalize='index')

In [ ]:
ct

In [ ]:
ct['Age'] = None

In [ ]:
for b in ct.index:
    if 'P2' in b:
        ct['Age'].loc[b] = 'P2/P3'  
    elif 'P3' in b:
        ct['Age'].loc[b] = 'P2/P3'
    elif 'P11' in b:
        ct['Age'].loc[b] = 'P11'
    else:# 'P60' in b:
        ct['Age'].loc[b] = 'P60'

In [ ]:
ctt = ct.reset_index().melt(id_vars=['batch'])[:-8]

ctt.rename({'value': 'Percentage of cells'}, axis=1, inplace=True)

ctt['Age'] = None

ctt = ctt.set_index(['batch'])

In [ ]:
for b in ctt.index:
    if 'P2' in b:
        ctt['Age'].loc[b] = 'P2/P3'  
    elif 'P3' in b:
        ctt['Age'].loc[b] = 'P2/P3'
    elif 'P11' in b:
        ctt['Age'].loc[b] = 'P11'
    else:# 'P60' in b:
        ctt['Age'].loc[b] = 'P60'

In [ ]:
from matplotlib.legend_handler import HandlerTuple

fig = plt.figure(figsize=(22,10))
ax = fig.add_subplot(111)    # The big subplot
ax1 = fig.add_subplot(211)
ax2 = fig.add_subplot(212)

# Turn off axis lines and ticks of the big subplot
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_color('none')
ax.spines['left'].set_color('none')
ax.spines['right'].set_color('none')
ax.tick_params(labelcolor='w', top=False, bottom=False, left=False, right=False)
ax.grid(False)

sns.boxplot(x = ctt['Cell_type'], y = ctt['Percentage of cells'], data = ctt, hue=ctt['Age'], 
            palette = my_c, showfliers = True, ax=ax1)

sns.stripplot(data=ctt, x='Cell_type', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], jitter=0.35, ax=ax1)

sns.boxplot(x = ctt['Cell_type'], y = ctt['Percentage of cells'], data = ctt, hue=ctt['Age'], 
            palette = my_c, showfliers = True, ax=ax2)

sns.stripplot(data=ctt, x='Cell_type', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], jitter=0.35155, ax=ax2)


ax1.set_ylim(0.80,1)   # those limits are fake
ax2.set_ylim(0,0.1)

sns.despine(ax=ax2)
sns.despine(ax=ax1, bottom=True)

d = .005  # how big to make the diagonal lines in axes coordinates

kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
ax1.plot((-d, +d), (-d, +d), **kwargs)        # top-left diagonal

kwargs.update(transform=ax2.transAxes)  # switch to the bottom axes
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # bottom-left diagonal

ax2.legend_.remove()
ax1.set(xlabel=None, ylabel=None)
ax2.set(xlabel=None,ylabel=None)
# ax1.legend_.remove()
ax1.set_xticks([])

ax1.set_yticks(np.arange(0.8,1.01, 0.05))
ax2.set_yticks(np.arange(0,0.11, 0.05))

from textwrap import wrap
names = [cell for cell in ctt['Cell_type'].unique()]

names = [ '\n'.join(wrap(l, 20)) for l in names ]

handles, labels = ax1.get_legend_handles_labels()
ax1.legend(handles=[(handles[0], handles[3]), (handles[1], handles[4]), (handles[2], handles[5])],
          labels=['P2/P3', 'P11', 'P60'], handlelength=2,
          handler_map={tuple: HandlerTuple(ndivide=None)},bbox_to_anchor=[1.12,1.05],fontsize=20)

ax2.set_xticklabels(names, fontsize=22)
# ax2.tick_params(names, labelsize=18)
ax1.tick_params(axis='y', labelsize=20)
ax2.tick_params(axis='y', labelsize=20)
ax.set_ylabel('Percantage of cells', fontsize=22)
ax.yaxis.set_label_coords(-0.05, 0.5)

plt
plt.show()

fig.savefig('boxplots_epithelial_all_celltypes.svg')

### Statistics on cell proportions

In [ ]:
ct_ttest = ct.copy()

In [ ]:
ct_ttest['Age'] = None

In [ ]:
for b in ct_ttest.index:
    if 'P2' in b:
        ct_ttest['Age'].loc[b] = 'P2/P3/P11'  
    elif 'P3' in b:
        ct_ttest['Age'].loc[b] = 'P2/P3/P11'
    elif 'P11' in b:
        ct_ttest['Age'].loc[b] = 'P2/P3/P11'
    else:# 'P60' in b:
        ct_ttest['Age'].loc[b] = 'P60'

In [ ]:
ctt_ttest = ct_ttest.reset_index().melt(id_vars=['batch'])[:-8]

In [ ]:
ctt_ttest.rename({'value': 'Percentage of cells'}, axis=1, inplace=True)

In [ ]:
ctt_ttest['Age'] = None

In [ ]:
ctt_ttest = ctt_ttest.set_index(['batch'])

In [ ]:
for b in ctt_ttest.index:
    if 'P2' in b:
        ctt_ttest['Age'].loc[b] = 'P2/P3/P11'  
    elif 'P3' in b:
        ctt_ttest['Age'].loc[b] = 'P2/P3/P11'
    elif 'P11' in b:
        ctt_ttest['Age'].loc[b] = 'P2/P3/P11'
    else:# 'P60' in b:
        ctt_ttest['Age'].loc[b] = 'P60'

In [ ]:
df_undone = {}

for ctype in ctt_ttest['Cell_type_w_cidea'].unique():
    for age in ctt_ttest['Age'].unique():
        print(ctype, age)
        sdf = ctt_ttest[(ctt_ttest['Cell_type_w_cidea'] == ctype) & (ctt_ttest['Age'] == age)]
        values = list(sdf['Percentage of cells'])
        df_undone[f"{ctype}_{age}"] = values


In [ ]:
from scipy.stats import ttest_ind

ttest_res = {}

for ctype in ctt_ttest['Cell_type_w_cidea'].unique():
    vals_list = []
    for k,v in df_undone.items():
        if ctype in k:
            print(k)
            vals_list.append(df_undone[k])
    f_statistic, p_value = ttest_ind(vals_list[0], vals_list[1])
    ttest_res[ctype] = [f_statistic, p_value]

In [ ]:
ttest_res_df = pd.DataFrame(ttest_res.items(), columns=['Cell_type_w_cidea', 'Values']).set_index('Cell_type_w_cidea')

In [ ]:
ttest_res_df['Values'].iloc[[0]]

In [ ]:
ttest_res_df['f_statistics'] = None
ttest_res_df['p_value'] = None

In [ ]:
for i in range(0, len(ttest_res_df['Values'])):
    ttest_res_df['f_statistics'].iloc[[i]] = ttest_res_df['Values'][i][0]
    ttest_res_df['p_value'].iloc[[i]] = ttest_res_df['Values'][i][1]

In [ ]:
ttest_res_df

In [ ]:
ttest_res_df.drop(columns='Values', inplace=True)
ttest_res_df

In [ ]:
ttest_res_df.to_csv('cell_proportions_statistics_epi.csv')

In [ ]:
from matplotlib.legend_handler import HandlerTuple

fig = plt.figure(figsize=(22,10))
ax = fig.add_subplot(111)    # The big subplot
ax1 = fig.add_subplot(211)
ax2 = fig.add_subplot(212)

# Turn off axis lines and ticks of the big subplot
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_color('none')
ax.spines['left'].set_color('none')
ax.spines['right'].set_color('none')
ax.tick_params(labelcolor='w', top=False, bottom=False, left=False, right=False)
ax.grid(False)

sns.boxplot(x = ctt_ttest['Cell_type_w_cidea'], y = ctt_ttest['Percentage of cells'], data = ctt_ttest, hue=ctt_ttest['Age'], 
            palette = my_c, showfliers = True, ax=ax1)

sns.stripplot(data=ctt_ttest, x='Cell_type_w_cidea', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], jitter=0.35, ax=ax1)

sns.boxplot(x = ctt_ttest['Cell_type_w_cidea'], y = ctt_ttest['Percentage of cells'], data = ctt_ttest, hue=ctt_ttest['Age'], 
            palette = my_c, showfliers = True, ax=ax2)

sns.stripplot(data=ctt_ttest, x='Cell_type_w_cidea', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], jitter=0.3551, ax=ax2)


ax1.set_ylim(0.8,1)   # those limits are fake
ax2.set_ylim(0,0.1)
sns.despine(ax=ax2)
sns.despine(ax=ax1, bottom=True)

d = .005  # how big to make the diagonal lines in axes coordinates

kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
ax1.plot((-d, +d), (-d, +d), **kwargs)        # top-left diagonal

kwargs.update(transform=ax2.transAxes)  # switch to the bottom axes
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # bottom-left diagonal

ax2.legend_.remove()
ax1.set(xlabel=None, ylabel=None)
ax2.set(xlabel=None,ylabel=None)
# ax1.legend_.remove()
ax1.set_xticks([])
ax1.set_yticks(np.arange(0.80,1, 0.1))
ax2.set_yticks(np.arange(0,0.11, 0.05))

from textwrap import wrap
names = [cell for cell in ctt_ttest['Cell_type_w_cidea'].unique()]

names = [ '\n'.join(wrap(l, 16)) for l in names ]

handles, labels = ax1.get_legend_handles_labels()
ax1.legend(handles=[(handles[0], handles[2]), (handles[1], handles[3])],
          labels=['P2/P3/P11', 'P60'], handlelength=2,
          handler_map={tuple: HandlerTuple(ndivide=None)},bbox_to_anchor=[1.12,1.05],fontsize=20)

ax2.set_xticklabels(names, fontsize=16)
# ax2.tick_params(names, labelsize=18)
ax1.tick_params(axis='y', labelsize=20)
ax2.tick_params(axis='y', labelsize=20)
ax.set_ylabel('Percantage of cells', fontsize=22)
ax.yaxis.set_label_coords(-0.05, 0.5)

plt
plt.show()

fig.savefig('ttest_boxplots_noDBL_all_celltypesgroups_epi.svg')

## Saving annotated data

In [ ]:
adata_noDBL.write_h5ad('noDBL_Epithelial_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

## Loading annotated data

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Epithelial_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
adata_noDBL

In [ ]:
adata_noDBL

In [ ]:
sc.pl.umap(adata_noDBL, color=['Cell_type', 'Age'], ncols=4, cmap='YlGnBu',wspace=0.8, palette=my_c)

# Gender inference based on Xist-counts and percent_chrY

In [ ]:
adata_noDBL = sc.read_h5ad('noDBL_Epithelial_Postnatal_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
adata_noDBL

In [ ]:
adata_noDBL.obs['Xist-counts']

In [ ]:
adata_noDBL.obs['percent_chrY']

In [ ]:
adata_noDBL.obs['Female'] = (adata_noDBL.obs['Xist-counts'] > 0)

In [ ]:
adata_noDBL.obs['Female']

In [ ]:
sc.pl.umap(adata_noDBL, color=['Cell_type', 'Female'], ncols=4, palette= my_c,wspace=0.7, save='_Celltypes_gender_epi.svg')

In [ ]:
sc.pl.umap(adata_noDBL, color=['percent_chrY', 'Xist-counts'], ncols=4, palette= my_c,wspace=0.7, vmax=0.2, save='_Percent_chrY_Xits_counts_epi.svg')